# 02 — Cleaning and preprocessing

Structural cleaning only: timestamps are parsed and sorted, duplicate timestamps are removed, infinities become NaN. No outlier trimming or global statistical imputation. Training-only imputation occurs inside the sklearn pipeline.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import RAW, PROCESSED, MODELS, ANALOG, DIGITAL, FAILURES

from src.data.load_data import load_raw
from src.preprocessing.preprocess import clean_data,temporal_split
raw=load_raw();df=clean_data(raw)
print('Before/after:',raw.shape,df.shape)
print('Remaining missing:',df.isna().sum().sort_values(ascending=False).head(20))

## Time gaps and unusual values

Large temporal gaps may represent logging outages, not sensor faults. Extreme values are flagged descriptively using robust statistics, never automatically discarded.

In [ ]:
delta=df['timestamp'].diff().dt.total_seconds()
display(delta.describe(percentiles=[.5,.9,.99,.999]))
print('Gaps > 120 s:',int((delta>120).sum()))
for c in [c for c in ANALOG if c in df]:
    median=df[c].median(); mad=(df[c]-median).abs().median()
    print(c,'robust extreme fraction (diagnostic only):',float(((df[c]-median).abs()>6*1.4826*mad).mean()) if mad>0 else 'undefined')

## Chronological partitions

Training ends March 31, validation covers April–May, test starts June 1. Do not derive preprocessing parameters from validation/test. Save structural cleaning as an optional reproducible intermediate.

In [ ]:
train,val,test=temporal_split(df)
for name,part in [('train',train),('validation',val),('test',test)]:print(name,len(part),part.timestamp.min(),part.timestamp.max())
PROCESSED.mkdir(parents=True,exist_ok=True)
df.to_pickle(PROCESSED/'cleaned.pkl')